# ClearTalk · fine-tune Whisper small.en on separated speech (Colab GPU)

Reproduces the local experiment on a Colab GPU: data → two-speaker mixtures → frozen SepFormer →
LoRA fine-tuning of Whisper small.en → validation → export for the app.

1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Run the cells from top to bottom. Expect several hours in total and ~25 GB of disk.
3. Upload `dist/ClearTalk_source.zip` (made locally with `python scripts/package_source.py`).

The test sets are **not** scored here: like the local run, the one-time final test belongs to
`ml_pipeline_english/04_eval.py` after the validation decision.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()  # choose ClearTalk_source.zip
zipfile.ZipFile(next(iter(uploaded))).extractall('/content/cleartalk')
os.chdir('/content/cleartalk')
print(sorted(os.listdir('.')))

In [ ]:
# Keep Colab's CUDA build of torch/torchaudio; pin everything else.
!grep -v -E '^(torch|torchaudio)==' requirements-train.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

In [ ]:
import json, pathlib, subprocess, tarfile, urllib.request
root = pathlib.Path('/content/cleartalk')
downloads = root / 'data/asr/downloads'
downloads.mkdir(parents=True, exist_ok=True)
openslr = 'https://www.openslr.org/resources'

def fetch(url, path):
    if not path.exists():
        subprocess.run(['curl', '-sSL', '--fail', '-o', str(path), url], check=True)

targets = {'train-clean-100': 'data/asr', 'dev-clean': 'data/asr', 'test-clean': 'data/asr',
           'test-other': 'data/asr', 'dev-other': 'data/v4'}
for name, destination in targets.items():
    archive = downloads / f'{name}.tar.gz'
    fetch(f'{openslr}/12/{name}.tar.gz', archive)
    with tarfile.open(archive) as tar:
        tar.extractall(root / destination)
    print('extracted', name)
pc = downloads / 'librispeech_pc_manifests.tar.gz'
fetch(f'{openslr}/145/manifests.tar.gz', pc)
(root / 'data/asr/librispeech_pc').mkdir(parents=True, exist_ok=True)
with tarfile.open(pc) as tar:
    tar.extractall(root / 'data/asr/librispeech_pc')
manifest = json.loads((root / 'data/v4/esc10/manifest.json').read_text())
for clip in manifest['clips']:
    path = root / clip['path']
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(clip['url'], path)
print('ESC-10 clips:', len(manifest['clips']))

In [ ]:
!python scripts/download_models.py --training

Steps 1–3: utterance table, profile and EDA figures (`ml_pipeline_english/figures`).

In [ ]:
!python ml_pipeline_english/01_eda.py

Step 5: render the frozen recipes in `data/asr/plans/` and separate them with SepFormer on the GPU.

In [ ]:
!python -m cleartalk_asr.mixtures build --set val --device cuda --worker colab
!python -m cleartalk_asr.mixtures build --set train --device cuda --worker colab

Steps 4–8 summary (uses the validation baseline in `data/asr/results/`, included in the zip).

In [ ]:
!python ml_pipeline_english/02_prep.py

Steps 9–12: LoRA fine-tuning with validation-based early stopping. On CUDA, bfloat16 autocast is safe and faster.

In [ ]:
!python -m cleartalk_asr.train --run colab --device cuda --precision bf16

In [ ]:
!python -m cleartalk_asr.export --adapter models/asr-runs/colab/best --output models/whisper-small.en-cleartalk
!cd models && zip -qr /content/whisper-small.en-cleartalk.zip whisper-small.en-cleartalk
files.download('/content/whisper-small.en-cleartalk.zip')

Unzip the download into the local project's `models/` folder, then run
`scripts/activate_model.py` there to compare it with the pretrained recognizer on validation.